# OCR DETECTION (NUXIA PROJECT):

## 1. Import libraries and dependancies

In [6]:
#pip install paddleocr
#pip install paddlepaddle

In [1]:
from paddleocr import PaddleOCR
from pathlib import Path
import cv2
import numpy as np
import shutil

### Data Structure:

/main 


//pruebas,//pruebas,//pruebas

//data_train (entrenar el classifier)
//data_all (divididas en troqueladas y no con sus previos análisis)

### Rutas:

In [ ]:
#original_path= "C:/Users/sampe/Nuxia" #change yours where you have the three folders

## 2. Classification model for die cut images

### 2.1. Pipeline to obtain the data to train the classifier

In [ ]:
from pathlib import Path
import shutil

CARPETA_ORIGEN = Path("pruebas14082026")
CARPETA_DESTINO = Path("data_train")
EXTENSIONES = {".png", ".jpg", ".jpeg"}

CARPETA_DESTINO.mkdir(exist_ok=True)

copiadas = 0
sin_original = []

for subcarpeta in sorted(p for p in CARPETA_ORIGEN.iterdir() if p.is_dir()):
    candidatas = [
        f for f in subcarpeta.iterdir()
        if f.stem.lower() == "original" and f.suffix.lower() in EXTENSIONES
    ]

    if not candidatas:
        sin_original.append(subcarpeta.name)
        continue

    origen = candidatas[0]
    destino_dir = CARPETA_DESTINO / subcarpeta.name
    destino_dir.mkdir(exist_ok=True)

    # copia la imagen (el original queda intacto)
    shutil.copy2(origen, destino_dir / f"original{origen.suffix.lower()}")

    # crea el txt vacío para etiquetar, sin pisar uno que ya hayas rellenado
    etiqueta = destino_dir / "etiqueta.txt"
    if not etiqueta.exists():
        etiqueta.touch()

    copiadas += 1


### 2.2 Pipeline to separate the die cut images from the one who doesn't

## 3. Image Preprocessing

In [9]:
ocr = PaddleOCR(
    lang="es",
    enable_mkldnn=False,                          # idioma (o "en", "ch", etc.)
    use_doc_orientation_classify=False, # desactiva módulos extra para ir más rápido
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

c:\Users\sampe\anaconda3\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sampe\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sampe\.paddlex\official_models\PP-OCRv6_medium_rec`.


In [20]:
path_img = Path("data_train/0e8ac78c")

In [21]:
img = cv2.imread(str(path_img / "original.png"))

In [26]:
resultado = ocr.predict(img)

for res in resultado:
    #res.print()                      # muestra todo en consola
    res.save_to_img(str(path_img / "ocr_no_pre.png"))        # imagen con las cajas dibujadas
    res.save_to_json(str(path_img / "report.json"))       # JSON con textos y coordenadas

### Let's try with different filters

In [ ]:
# GRIS

gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
imagen_limpia = clahe.apply(gris)
cv2.imwrite("salida/gris.png", imagen_limpia)

imagen_3c = cv2.cvtColor(imagen_limpia, cv2.COLOR_GRAY2BGR) # Necesitas tres canales ya que Paddle OCR lo requiere

for res in ocr.predict(imagen_3c):
    res.print()
    res.save_to_img("salida")        # imagen con las cajas dibujadas
    res.save_to_json("salida") 

True

In [ ]:
# CLAHE

lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)
l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)

imagen_3c = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

cv2.imwrite("salida/clahe.png", imagen_3c)

for res in ocr.predict(imagen_3c):
    res.print()
    res.save_to_img("salida")        # imagen con las cajas dibujadas
    res.save_to_json("salida") 

In [ ]:
# ADAPTATIVE

gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
bw = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY, 31, 15)

imagen_3c = cv2.cvtColor(bw, cv2.COLOR_GRAY2BGR) 

cv2.imwrite("salida/adaptative.png", imagen_3c)

for res in ocr.predict(imagen_3c):
    res.print()
    res.save_to_img("salida")        # imagen con las cajas dibujadas
    res.save_to_json("salida") 


In [ ]:

def brillo_contraste(img, alpha=1.3, beta=20):
    # alpha: contraste (1.0 = igual), beta: brillo (-100..100)
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)


def gamma(img, g=0.6):
    # g < 1 aclara, g > 1 oscurece
    tabla = np.array([(i / 255.0) ** g * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(img, tabla)



def normalizar(img):
    return cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)


## 4. Image Information